In [3]:
from xgboost import XGBClassifier as xgb 
from lightgbm import LGBMClassifier as lgbm
from catboost import CatBoostClassifier as cbc 
from sklearn.ensemble import RandomForestClassifier as rfc

In [4]:
from sklearn.model_selection import train_test_split as tts
from sklearn.pipeline import Pipeline as p
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV

In [10]:
dataset=load_iris(as_frame=True)
y=dataset.target
x=dataset.data

In [12]:
x_train, x_test, y_train, y_test = tts(x, y,test_size=0.2,random_state=18)

In [16]:
param_grid = {

    "Random Forest": {
        "model": rfc(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [None, 5, 10],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2]
        }
    },

    "XGBoost": {
        "model": xgb(
            random_state=42,
            eval_metric="mlogloss"
        ),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 0.3],
            "max_depth": [3, 5, 7]
        }
    },

    "LightGBM": {
        "model": lgbm(
            random_state=42,
            verbose=-1
        ),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 0.3],
            "max_depth": [-1, 5, 10]
        }
    },

    "CatBoost": {
        "model": cbc(
            random_state=42,
            verbose=0,
            allow_writing_files=False
        ),
        "params": {
            "iterations": [50, 100, 200],
            "learning_rate": [0.01, 0.1, 0.3],
            "depth": [4, 6, 8]
        }
    }

}

In [21]:
best_models = {}
scores = []

for model_name, mp in param_grid.items():

    g = GridSearchCV(
        mp["model"],
        mp["params"],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    g.fit(x_train, y_train)

    best_models[model_name] = g.best_estimator_

    scores.append({
        "Model": model_name,
        "Best Score": g.best_score_,
        "Best Params": g.best_params_
    })

In [22]:
import pandas as pd

In [23]:
pd.DataFrame(scores)

,Model,Best Score,Best Params
0,Random Forest,0.950000,"{'max_depth': None, 'min_samples_leaf': 1, 'mi..."
1,XGBoost,0.950000,"{'learning_rate': 0.01, 'max_depth': 3, 'n_est..."
2,LightGBM,0.941667,"{'learning_rate': 0.01, 'max_depth': -1, 'n_es..."
3,CatBoost,0.958333,"{'depth': 4, 'iterations': 50, 'learning_rate'..."


In [24]:
best_models

{'Random Forest': RandomForestClassifier(min_samples_split=5, n_estimators=50, random_state=42),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric='mlogloss',
               feature_types=None, feature_weights=None, gamma=None,
               grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.01, max_bin=None,
               max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=3, max_leaves=None,
               min_child_weight=None, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=50, n_jobs=None,
               num_parallel_tree=None, ...),
 'LightGBM': LGBMClassifier(learning_rate=0.01, random_state=42, verbose=-1),
 'CatBoost': CatBoostClass

In [25]:
rf=best_models['Random Forest']
xg=best_models['XGBoost']
lg=best_models['LightGBM']
cb=best_models['CatBoost']

In [32]:
rf.score(x_test,y_test)

1.0

In [33]:
xg.score(x_test,y_test)

1.0

In [34]:
lg.score(x_test,y_test)

0.9333333333333333

In [35]:
cb.score(x_test,y_test)

np.float64(1.0)